# Customer Segmentation Using K-Means Clustering
**AIML Minor Project — Boosting / Unsupervised Learning**

**Dataset:** Mall Customer Segmentation Data (Kaggle)  
**Kaggle link:** https://www.kaggle.com/datasets/vjchoudhary7/customer-segmentation-tutorial-in-python

## 1. Problem statement
A mall has basic membership information about customers but needs a clearer understanding of different spending behaviours. This project groups customers into segments using age, annual income, and spending score, then describes the segments to support more relevant marketing ideas.

## Objectives
- Inspect data quality and document cleaning decisions.
- Explore customer demographics, income, and spending score.
- Use K-Means clustering to identify customer groups.
- Select a cluster count using inertia (elbow) and silhouette score.
- Profile clusters and translate findings into cautious business recommendations.

**Important:** This is an unsupervised-learning task. There is no ground-truth segment label, so classification accuracy is not an appropriate metric. Internal metrics and interpretability are used instead.

## 2. Dataset and setup
The Kaggle dataset is commonly distributed as `Mall_Customers.csv`. It contains customer ID, gender, age, annual income (in thousands of dollars), and a spending score (1–100) assigned by the mall. Download the CSV from the Kaggle page above and place it in the same folder as this notebook.

The Kaggle page describes the data as an educational customer-segmentation dataset. Confirm the dataset's current licence and usage terms on Kaggle before redistribution.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

# Place Mall_Customers.csv beside this notebook.
DATA_PATH = "Mall_Customers.csv"
df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
display(df.head())

## 3. Understand the data

In [ ]:
print("Rows, columns:", df.shape)
print("\nColumn names:", df.columns.tolist())
print("\nData types and non-null counts:")
df.info()
print("\nSummary statistics:")
display(df.describe(include="all").T)

## 4. Data quality checks and cleaning

In [ ]:
print("Missing values per column:")
display(df.isna().sum().to_frame("missing_count"))
print("Exact duplicate rows:", df.duplicated().sum())

# Standardize column labels so later code is easier to read.
rename_map = {
    "CustomerID": "customer_id",
    "Gender": "gender",
    "Age": "age",
    "Annual Income (k$)": "annual_income_k",
    "Spending Score (1-100)": "spending_score"
}
df = df.rename(columns=rename_map)

required = ["customer_id", "gender", "age", "annual_income_k", "spending_score"]
missing_cols = [c for c in required if c not in df.columns]
if missing_cols:
    raise ValueError(f"Expected columns not found: {missing_cols}. Check the CSV column names.")

# Preserve original row count for reporting; remove exact duplicates and rows missing key values.
rows_before = len(df)
df = df.drop_duplicates().copy()
df = df.dropna(subset=["age", "annual_income_k", "spending_score"]).copy()

# Validate ranges; these are data checks, not claims about all possible customers.
valid = (
    df["age"].between(1, 100) &
    df["annual_income_k"].gt(0) &
    df["spending_score"].between(1, 100)
)
invalid_count = int((~valid).sum())
df = df.loc[valid].copy()

print(f"Rows before cleaning: {rows_before}")
print(f"Rows after cleaning:  {len(df)}")
print(f"Rows removed as duplicates/missing/invalid: {rows_before-len(df)}")
print(f"Invalid-range rows detected before removal: {invalid_count}")
print("\nMissing values after cleaning:")
display(df.isna().sum().to_frame("missing_after_cleaning"))

**Cleaning decisions**
- `customer_id` is an identifier, not a behavioural feature, so it is excluded from clustering.
- Exact duplicate rows are removed.
- Rows missing age, income, or spending score are removed because these are required for the chosen analysis.
- Clearly implausible values are flagged using broad validity checks. Review flagged rows before removing them in a real business setting.
- Gender is retained for descriptive comparison, but excluded from the primary clustering features to focus segmentation on behavioural/demographic numeric variables and avoid arbitrary category encoding.

## 5. Exploratory data analysis (EDA)

In [ ]:
# Visualization 1: customer age distribution
plt.figure(figsize=(8, 4.5))
sns.histplot(df["age"], bins=15, kde=True)
plt.title("Customer Age Distribution")
plt.xlabel("Age (years)")
plt.ylabel("Number of customers")
plt.tight_layout()
plt.show()

**Observation 1:** Use the histogram to identify the age ranges most represented in this sample. Do not assume the sample represents every mall customer.

In [ ]:
# Visualization 2: annual income distribution
plt.figure(figsize=(8, 4.5))
sns.histplot(df["annual_income_k"], bins=15, kde=True)
plt.title("Annual Income Distribution")
plt.xlabel("Annual income (thousands of dollars)")
plt.ylabel("Number of customers")
plt.tight_layout()
plt.show()

**Observation 2:** The income histogram shows how the sample is spread across income levels and whether a few observations sit at the extremes.

In [ ]:
# Visualization 3: spending score by age
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x="age", y="spending_score", hue="gender", alpha=.8)
plt.title("Age vs Spending Score")
plt.xlabel("Age (years)")
plt.ylabel("Spending score (1–100)")
plt.tight_layout()
plt.show()

**Observation 3:** Look for broad patterns and overlap. A visible association does not prove that age causes spending behaviour.

In [ ]:
# Visualization 4: correlation heatmap
numeric_cols = ["age", "annual_income_k", "spending_score"]
plt.figure(figsize=(7, 5))
sns.heatmap(df[numeric_cols].corr(), annot=True, cmap="coolwarm", center=0, fmt=".2f")
plt.title("Correlation Between Numeric Customer Features")
plt.tight_layout()
plt.show()

**Observation 4:** Correlations summarize linear relationships only. Weak correlation does not rule out useful non-linear or group-level patterns.

## 6. Feature selection and scaling

In [ ]:
features = ["annual_income_k", "spending_score"]
X = df[features].copy()

# K-Means uses distances; standardize features so scale differences do not dominate.
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Features used:", features)
print("Scaled feature means (approximately 0):", X_scaled.mean(axis=0).round(3))
print("Scaled feature standard deviations (approximately 1):", X_scaled.std(axis=0).round(3))

## 7. Select the number of clusters

In [ ]:
candidate_k = range(2, 11)
inertias, silhouettes = [], []

for k in candidate_k:
    model = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=20)
    labels = model.fit_predict(X_scaled)
    inertias.append(model.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))

metrics_table = pd.DataFrame({
    "k": list(candidate_k),
    "inertia": inertias,
    "silhouette_score": silhouettes
})
display(metrics_table)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(metrics_table["k"], metrics_table["inertia"], marker="o")
axes[0].set_title("Elbow Method")
axes[0].set_xlabel("Number of clusters (k)")
axes[0].set_ylabel("Inertia (within-cluster SSE)")
axes[1].plot(metrics_table["k"], metrics_table["silhouette_score"], marker="o")
axes[1].set_title("Silhouette Score by k")
axes[1].set_xlabel("Number of clusters (k)")
axes[1].set_ylabel("Silhouette score")
plt.tight_layout()
plt.show()

**How to choose k:** The elbow plot shows diminishing returns in within-cluster error as k increases. The silhouette score measures how separated and compact the clusters are; higher is generally better, but it should not be used alone. Choose a value that balances these measures with clusters that are understandable and useful. The next cell uses the highest silhouette score among tested k values as a reproducible baseline; if the elbow and business interpretation suggest another value, explain and document that choice.

In [ ]:
best_row = metrics_table.loc[metrics_table["silhouette_score"].idxmax()]
chosen_k = int(best_row["k"])
print("Baseline k selected by highest silhouette score:", chosen_k)

## 8. Train final K-Means model and visualize clusters

In [ ]:
kmeans = KMeans(n_clusters=chosen_k, random_state=RANDOM_STATE, n_init=20)
df["cluster"] = kmeans.fit_predict(X_scaled)

# Visualization 5: cluster map
plt.figure(figsize=(9, 6))
sns.scatterplot(
    data=df, x="annual_income_k", y="spending_score",
    hue="cluster", palette="tab10", s=75, alpha=.9
)
centers_original = scaler.inverse_transform(kmeans.cluster_centers_)
plt.scatter(centers_original[:, 0], centers_original[:, 1],
            marker="X", s=250, c="black", label="Cluster centers")
plt.title(f"Customer Segments from K-Means (k={chosen_k})")
plt.xlabel("Annual income (thousands of dollars)")
plt.ylabel("Spending score (1–100)")
plt.legend(title="Segment", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

## 9. Evaluate and profile the segments

In [ ]:
final_silhouette = silhouette_score(X_scaled, df["cluster"])
print(f"Final silhouette score: {final_silhouette:.3f}")
print(f"Final inertia: {kmeans.inertia_:.3f}")

profile = df.groupby("cluster").agg(
    customers=("customer_id", "count"),
    mean_age=("age", "mean"),
    mean_income_k=("annual_income_k", "mean"),
    mean_spending_score=("spending_score", "mean"),
    median_income_k=("annual_income_k", "median"),
    median_spending_score=("spending_score", "median")
).round(2)
profile["customer_share_pct"] = (profile["customers"] / len(df) * 100).round(1)
display(profile.sort_values("mean_spending_score", ascending=False))

**Evaluation interpretation**
- **Silhouette score:** ranges from -1 to +1. Values nearer +1 indicate better separation; values around 0 suggest overlapping clusters; negative values can indicate questionable assignments.
- **Inertia:** lower values mean points are closer to their assigned centroids, but inertia always tends to decrease as k grows, so it cannot choose k by itself.
- There is no true label in this dataset, so accuracy, precision, recall, and F1-score are not applicable.

In [ ]:
# Visualization 6: segment sizes
cluster_sizes = df["cluster"].value_counts().sort_index()
plt.figure(figsize=(8, 4.5))
sns.barplot(x=cluster_sizes.index.astype(str), y=cluster_sizes.values)
plt.title("Number of Customers in Each Segment")
plt.xlabel("Cluster")
plt.ylabel("Number of customers")
plt.tight_layout()
plt.show()

# Export the customer-level segment labels and the segment profile.
df.to_csv("customer_segments_output.csv", index=False)
profile.to_csv("segment_profile.csv")
print("Saved customer_segments_output.csv and segment_profile.csv")

## 10. Business interpretation and recommendations

Use the actual cluster profile table and scatter plot to label segments by behaviour, not by stereotypes. Common patterns may include:
- **High income / high spending:** consider loyalty benefits and early access to relevant offers.
- **High income / low spending:** test personalized engagement or value propositions; do not assume the reason for low spending.
- **Lower income / high spending:** consider affordable bundles and loyalty rewards.
- **Lower income / low spending:** test low-cost promotions and measure whether they improve engagement.
- **Middle or mixed segments:** tailor offers only when the profile table supports a meaningful distinction.

These labels are examples. Assign final segment names only after examining the calculated cluster centres and profile table. Test recommendations with measured outcomes before deploying campaigns.

## 11. Limitations

1. The dataset is small (commonly 200 rows) and is a learning dataset, not necessarily representative of a real mall.
2. Spending score is assigned by the mall and may not be directly comparable to actual transaction value.
3. The data contains limited customer attributes and no transaction history, product preferences, customer lifetime value, or time dimension.
4. K-Means favours compact, roughly spherical clusters and is sensitive to feature selection and scaling.
5. Cluster numbers are arbitrary labels; Cluster 0 is not inherently better than Cluster 1.
6. Results describe patterns in this sample and do not establish causal relationships.

## 12. Conclusion

This project demonstrates a complete unsupervised-learning workflow: data-quality checks, exploratory visualization, feature scaling, selection of cluster count, K-Means training, internal evaluation, and segment profiling. The final conclusion should quote the computed silhouette score, chosen k, cluster sizes, and the distinguishing features from the profile table after running the notebook on the downloaded Kaggle CSV. Recommendations should be treated as hypotheses to test rather than guaranteed outcomes.

## References
- Kaggle dataset: https://www.kaggle.com/datasets/vjchoudhary7/customer-segmentation-tutorial-in-python
- scikit-learn K-Means / clustering documentation: https://scikit-learn.org/stable/modules/clustering.html
- scikit-learn StandardScaler documentation: https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html
- scikit-learn silhouette analysis: https://scikit-learn.org/stable/auto_examples/cluster/plot_kmeans_silhouette_analysis.html